# **Project Overview**

- This project involves an **Exploratory Data Analysis (EDA)** of a **Superstore dataset** using **SQL**.
- The core objective is to uncover insights into **sales**, **customer behavior**, and **product performance**.
- This is achieved through a multi-step process:
  - Data preprocessing and normalization to create a well-structured database.
  - Calculating **Key Performance Indicators (KPIs)**.
  - Conducting detailed **sales**, **customer**, and **product** analyses using various SQL queries.
- The project aims to provide **actionable business intelligence** from the raw transactional data.
- This project leverages several **fundamental and advanced SQL concepts** including:
  - Aggregate functions  
  - String functions  
  - Window functions  
  - Joins  
  - Common Table Expressions (CTEs)  
  - Conditional logic  



---

# Superstore Data Preprocessing (Normalization & Cleaning)

This document outlines the step-by-step process for cleaning and normalizing the `superstore` dataset using SQL.

---

## Database Setup

```sql
USE `superstore analysis`;
```

---

## 1. Creating `location lookup` Table

Normalize `location`-related fields from the `superstore` table.

```sql
CREATE TABLE `location lookup` AS 
SELECT 
	`country`, `state`, `city`, `postal code`
FROM `superstore`
GROUP BY `country`, `state`, `city`, `postal code`
ORDER BY `country`, `state`, `city`, `postal code`;
```

Add `Location Id`

```sql
ALTER TABLE `location lookup`
ADD COLUMN `Location Id` VARCHAR(10);
```

Populate the `Location Id` column:

```sql
UPDATE `location lookup` AS `loc lookup`
SET `location id` = (
	SELECT CONCAT("L", rn)
	FROM (
		SELECT 
			`country`, `state`, `city`, `postal code`, 
			ROW_NUMBER() OVER () AS rn
		FROM `location lookup`
		GROUP BY `country`, `state`, `city`, `postal code`
		ORDER BY `country`, `state`, `city`, `postal code`
	) AS derived
	WHERE
		`loc lookup`.`country` = `derived`.`country` AND 
		`loc lookup`.`state` = `derived`.`state` AND 
		`loc lookup`.`city` = `derived`.`city` AND 
		`loc lookup`.`postal code` = `derived`.`postal code`
);
```

Set `Location Id` as Primary Key:

```sql
ALTER TABLE `location lookup`
ADD PRIMARY KEY (`location id`);
```

---

Add `region` to `location lookup`

Check for multiple regions per location:

```sql
SELECT 
	`location id`, 
    COUNT(DISTINCT `region`) AS `distinct regions`
FROM `superstore`
GROUP BY `location id`
HAVING `distinct regions` > 1;
```

Update the `location lookup` table:

```sql
ALTER TABLE `location lookup`
ADD COLUMN `region` TEXT;

UPDATE `location lookup` AS `loc`
SET `region` = (
	SELECT DISTINCT `region`
	FROM `superstore` AS `super`
	WHERE `loc`.`location id` = `super`.`location id`
);
```

Rearrange columns:

```sql
ALTER TABLE `location lookup`
	MODIFY COLUMN `location id` VARCHAR(10) FIRST;

ALTER TABLE `location lookup`
	MODIFY COLUMN `region` TEXT AFTER `country`;
```

---

## 2. Creating `order lookup` Table

Check uniqueness:

```sql
SELECT 
	`order id`, 
    COUNT(DISTINCT `ship mode`) AS `distinct values`
FROM `superstore`
GROUP BY `order id`
HAVING `distinct values` > 1;
```

Create `order lookup`:

```sql
CREATE TABLE `order lookup` AS 
SELECT 
	`order id`, `order date`, `ship date`, `ship mode`
FROM `superstore`
GROUP BY `order id`, `order date`, `ship date`, `ship mode`
ORDER BY `order id`, `order date`, `ship date`, `ship mode`;
```

Format data types and set primary key:

```sql
ALTER TABLE `order lookup`
MODIFY COLUMN `order id` VARCHAR(20);

UPDATE `order lookup`
SET `order date` = STR_TO_DATE(`order date`, '%d-%m-%Y');

UPDATE `order lookup`
SET `ship date` = STR_TO_DATE(`ship date`, '%d-%m-%Y');

ALTER TABLE `order lookup`
ADD PRIMARY KEY (`order id`);
```

---

## 3. Preprocessing `customer lookup`

```sql
ALTER TABLE `customer lookup`
MODIFY COLUMN `customer id` VARCHAR(10);

ALTER TABLE `customer lookup`
ADD PRIMARY KEY (`customer id`);
```

---

## 4. Preprocessing `product lookup`

Handle duplicates in `product id`:

```sql
SELECT 
	`product id`, `category`, `sub-category`, 
    COUNT(*) 
FROM `product lookup`
GROUP BY `product id`, `category`, `sub-category`
HAVING COUNT(*) > 1;
```

Create a cleaned version:

```sql
CREATE TABLE `product lookup clone` AS 
SELECT
	`product id` AS `Product Id`, 
    `category` AS `Category`, 
    `sub-category` AS `Sub-Category`, 
    GROUP_CONCAT(`product name` SEPARATOR ", ") AS `Product Name`
FROM `product lookup`
GROUP BY `product id`, `category`, `sub-category`;

DROP TABLE `product lookup`;

CREATE TABLE `product lookup` AS 
SELECT * FROM `product lookup clone`;

ALTER TABLE `product lookup`
MODIFY COLUMN `product id` VARCHAR(20);

ALTER TABLE `product lookup`
ADD PRIMARY KEY (`product id`);

DROP TABLE `product lookup clone`;
```

---

## 5. Preprocessing `superstore` Table

Update Data Types

```sql
ALTER TABLE `superstore`
MODIFY COLUMN `customer id` VARCHAR(10),
MODIFY COLUMN `product id` VARCHAR(20),
MODIFY COLUMN `order id` VARCHAR(20);
```

Add `Profit or Loss` Column

```sql
ALTER TABLE `superstore`
ADD COLUMN `Profit or Loss` VARCHAR(10); 

UPDATE `superstore`
SET `Profit or Loss` =
	CASE
		WHEN profit < 0 THEN "Loss"
		WHEN profit > 0 THEN "Profit"
		ELSE "None"
	END;
```

### Add `Location Id` and Drop Redundant Fields

```sql
ALTER TABLE `superstore`
ADD COLUMN `Location Id` VARCHAR(10);

UPDATE `superstore` `super`
SET `location id` = (
	SELECT `location id`
	FROM `location lookup` AS `loc`
	WHERE
		`super`.`country` = `loc`.`country` AND 
		`super`.`state` = `loc`.`state` AND 
		`super`.`city` = `loc`.`city` AND 
		`super`.`postal code` = `loc`.`postal code`
);

ALTER TABLE `superstore`
	DROP COLUMN `country`, 
	DROP COLUMN `state`, 
	DROP COLUMN `city`, 
	DROP COLUMN `postal code`,
	DROP COLUMN `region`,
	DROP COLUMN `order date`, 
	DROP COLUMN `ship date`, 
	DROP COLUMN `ship mode`;
```

Rearrange column:

```sql
ALTER TABLE `superstore`
MODIFY COLUMN `location id` VARCHAR(10) AFTER `segment`;
```

---

## 6. Defining Relationships (Foreign Keys)

```sql
ALTER TABLE `superstore`
ADD CONSTRAINT fk_superstore_customer
FOREIGN KEY (`customer id`)
REFERENCES `customer lookup`(`customer id`) ON DELETE CASCADE;

ALTER TABLE `superstore`
ADD CONSTRAINT fk_superstore_product
FOREIGN KEY (`product id`)
REFERENCES `product lookup`(`product id`) ON DELETE CASCADE;

ALTER TABLE `superstore`
ADD CONSTRAINT fk_superstore_location
FOREIGN KEY (`location id`)
REFERENCES `location lookup`(`location id`) ON DELETE CASCADE;

ALTER TABLE `superstore`
ADD CONSTRAINT fk_superstore_order
FOREIGN KEY (`order id`)
REFERENCES `order lookup`(`order id`) ON DELETE CASCADE;
```

---

## Summary

This preprocessing pipeline includes:

* **Normalization** of the `superstore` dataset into multiple lookup tables
* **Data type formatting**
* **Removal of redundant columns**
* **Creation of unique IDs and primary keys**
* **Establishing relational integrity** via foreign key constraints

This approach improves **data consistency**, supports **scalability**, and aligns with **data warehousing best practices**.

---




---

# Key Performance Indicators (KPIs) 

This section highlights essential KPIs calculated from the `superstore` dataset.

---

## 1. Total Sales

```sql
SELECT ROUND(SUM(`sales`), 2) AS `Total Sales`
FROM `superstore`;
```

**Description**: Calculates the **total revenue** generated from all transactions.

---

## 2. Total Orders

```sql
SELECT COUNT(DISTINCT `order id`) 
FROM `superstore`;
```

**Description**: Counts the **number of unique orders** placed.

---

## 3. Total Profit

```sql
SELECT ROUND(SUM(`profit`), 2)
FROM `superstore`
WHERE `profit or loss` = "Profit";
```

**Description**: Computes the **sum of all profitable transactions**.

---

## 4. Total Loss

```sql
SELECT ABS(ROUND(SUM(`profit`), 2))
FROM `superstore`
WHERE `profit or loss` = "Loss";
```

**Description**: Computes the **absolute value of total loss**, where profit is negative.

---




---

# Sales Analysis 

This section provides an overview of sales trends and profitability across time, based on the `superstore` dataset.

---

## 1. Order Categorization by Profit Status

```sql
SELECT 
	`profit or loss` AS `Category`,
	COUNT(DISTINCT `order id`) AS `Distinct Order ids`, 
	CONCAT(ROUND(COUNT(DISTINCT `order id`) * 100 / 
        (SELECT COUNT(DISTINCT `order id`) FROM `superstore`), 2), "%") AS `Percentage proportion`
FROM `superstore` 
GROUP BY `Category`;
```

| Category | Distinct Order IDs | Percentage Proportion |
| -------- | ------------------ | --------------------- |
| Loss     | 1318               | 26.31%                |
| None     | 64                 | 1.28%                 |
| Profit   | 4407               | 87.98%                |

> ⚠ **Note**: The total exceeds 100% because some orders include multiple line items with mixed profit categories (e.g., one item profitable, another at a loss or neutral).

another method to achieve the same result

```sql
SELECT DISTINCT `order id` 
FROM `superstore` 
WHERE `profit or loss` = "Profit"
INTERSECT
SELECT DISTINCT `order id` 
FROM `superstore` 
WHERE `profit or loss` = "Loss"
INTERSECT
SELECT DISTINCT `order id` 
FROM `superstore` 
WHERE `profit or loss` = "None";
```

---

## 2. Total Sales by Year

```sql
SELECT 
	YEAR(`ol`.`order date`) AS `Year`, 
	ROUND(SUM(`s`.`sales`), 2) AS `Total Sales`
FROM `superstore` `s`
JOIN `order lookup` `ol` ON `s`.`order id` = `ol`.`order id`
GROUP BY `Year`
ORDER BY `Year`;
```

| Year | Total Sales |
| ---- | ----------- |
| 2014 | 484,247.50  |
| 2015 | 470,532.51  |
| 2016 | 609,205.60  |
| 2017 | 733,215.26  |

---

## 3. Year-Over-Year (YOY) Sales Growth

```sql
WITH cte AS (
	SELECT 
		YEAR(`ol`.`order date`) AS `Year`, 
		ROUND(SUM(`s`.`sales`), 2) AS `Total Sales`
	FROM `superstore` `s`
	JOIN `order lookup` `ol` ON `s`.`order id` = `ol`.`order id`
	GROUP BY `Year`
	ORDER BY `Year`
)
SELECT 
	`c2`.`Year` AS `Current Year`, 
	`c1`.`Year` AS `Previous Year`, 
	CONCAT(ROUND((`c2`.`Total Sales` - `c1`.`Total Sales`) / `c1`.`Total Sales` * 100, 2), "%") AS `YOY Sales Growth`
FROM `cte` `c1`
JOIN `cte` `c2` ON `c1`.`Year` = `c2`.`Year` - 1;
```

| Current Year | Previous Year | YOY Sales Growth |
| ------------ | ------------- | ---------------- |
| 2015         | 2014          | -2.83%           |
| 2016         | 2015          | 29.47%           |
| 2017         | 2016          | 20.36%           |

---




---

# Customer Analysis 

This section focuses on identifying top customers, regional contributions, and customer buying patterns using SQL queries from the `superstore` dataset.

---

## 1. Top Customers by Total Purchase Amount

```sql
SELECT 
	t2.`customer id` AS `Customer Id`,
	t2.`customer name` AS `Customer Name`, 
	ROUND(SUM(`sales`), 2) AS `Total Purchase Amount`
FROM `superstore` AS t1
LEFT JOIN `customer lookup` AS t2 ON t1.`customer id` = t2.`customer id`
GROUP BY `Customer Id`, `Customer Name`
ORDER BY `Total Purchase Amount` DESC 
LIMIT 5;
```

| Customer Id | Customer Name | Total Purchase Amount |
| ----------- | ------------- | --------------------- |
| SM-20320    | Sean Miller   | 25,043.05             |
| TC-20980    | Tamara Chand  | 19,052.22             |
| RB-19360    | Raymond Buch  | 15,117.34             |
| TA-21385    | Tom Ashbrook  | 14,595.62             |
| AB-10105    | Adrian Barton | 14,473.57             |

---

## 2. Top Customers by Region

```sql
WITH cte AS (
	SELECT 
		`ll`.`region` AS `Region`, 
		`cl`.`customer id` AS `Customer Id`, 
		`cl`.`customer name` AS `Customer Name`,
		`s`.`sales` AS `Total Sales`, 
		ROW_NUMBER() OVER (PARTITION BY `ll`.`region` ORDER BY `s`.`sales` DESC) AS `Rank`
	FROM `superstore` AS `s`
	JOIN `location lookup` AS `ll` ON `s`.`location id` = `ll`.`location id`
	JOIN `customer lookup` AS `cl` ON `s`.`customer id` = `cl`.`customer id`
)
SELECT `Region`, `Customer Id`, `Customer Name`, `Total Sales`, `Rank`
FROM cte
WHERE `Rank` = 1;
```

| Region  | Customer Id | Customer Name | Total Sales | Rank |
| ------- | ----------- | ------------- | ----------- | ---- |
| Central | TC-20980    | Tamara Chand  | 17,499.95   | 1    |
| East    | TA-21385    | Tom Ashbrook  | 11,199.97   | 1    |
| South   | SM-20320    | Sean Miller   | 22,638.48   | 1    |
| West    | RB-19360    | Raymond Buch  | 13,999.96   | 1    |

---

## 3. Regional Contribution to Orders

```sql
SELECT 
	`ll`.`region` AS `Region`, 
	COUNT(DISTINCT `ol`.`order id`) AS `Order Count`, 
	CONCAT(ROUND((COUNT(DISTINCT `ol`.`order id`) * 100) / 
        (SELECT COUNT(*) FROM `order lookup`), 2), "%") AS `Percentage of Regions Contributing to Orders`
FROM `superstore` AS `ss`
JOIN `location lookup` AS `ll` ON `ss`.`location id` = `ll`.`location id`
JOIN `order lookup` AS `ol` ON `ss`.`order id` = `ol`.`order id`
GROUP BY `Region`;
```

| Region  | Order Count | % Contribution to Orders |
| ------- | ----------- | ------------------------ |
| Central | 1175        | 23.46%                   |
| East    | 1401        | 27.97%                   |
| South   | 822         | 16.41%                   |
| West    | 1611        | 32.16%                   |

---

## 4. Customers Who Frequently Buy the Same Product Category

```sql
SELECT 
	`ss`.`customer id` AS `Customer Id`, 
	`cl`.`customer name` AS `Customer Name`,
	`pl`.`category` AS `Product Category`, 
	COUNT(*) AS `Order Quantity`
FROM `superstore` AS `ss`
JOIN `product lookup` AS `pl` ON `ss`.`product id` = `pl`.`product id`
JOIN `customer lookup` AS `cl` ON `ss`.`customer id` = `cl`.`customer id`
GROUP BY `Customer Id`, `Product Category`
HAVING `Order Quantity` > 18
ORDER BY `Order Quantity` DESC;
```

| Customer Id | Customer Name       | Product Category | Order Quantity |
| ----------- | ------------------- | ---------------- | -------------- |
| EH-13765    | Edward Hooks        | Office Supplies  | 26             |
| WB-21850    | William Brown       | Office Supplies  | 23             |
| JD-15895    | Jonathan Doherty    | Office Supplies  | 22             |
| AP-10915    | Arthur Prichep      | Office Supplies  | 21             |
| MA-17560    | Matt Abelman        | Office Supplies  | 21             |
| GT-14710    | Greg Tran           | Office Supplies  | 21             |
| XP-21865    | Xylona Preis        | Office Supplies  | 21             |
| CK-12205    | Chloris Kastensmidt | Office Supplies  | 21             |
| CS-12250    | Chris Selesnick     | Office Supplies  | 21             |
| CB-12025    | Cassandra Brandow   | Office Supplies  | 20             |
| JL-15835    | John Lee            | Office Supplies  | 20             |
| DK-12835    | Damala Kotsonis     | Office Supplies  | 20             |
| SH-19975    | Sally Hughsby       | Office Supplies  | 19             |
| BM-11650    | Brian Moss          | Office Supplies  | 19             |
| PP-18955    | Paul Prost          | Office Supplies  | 19             |
| RP-19390    | Resi Pölking        | Office Supplies  | 19             |
| EP-13915    | Emily Phan          | Office Supplies  | 19             |

---

## 5. Most Active Customers by Number of Orders

```sql
SELECT 
	`cl`.`customer id` AS `Customer Id`,
	`cl`.`customer name` AS `Customer Name`, 
	COUNT(DISTINCT `order id`) AS `Total Orders`
FROM `superstore` `ss`
JOIN `customer lookup` `cl` ON `ss`.`customer id` = `cl`.`customer id`
GROUP BY `customer id`
ORDER BY `Total Orders` DESC
LIMIT 5;
```

| Customer Id | Customer Name       | Total Orders |
| ----------- | ------------------- | ------------ |
| EP-13915    | Emily Phan          | 17           |
| ZC-21910    | Zuschuss Carroll    | 13           |
| PG-18820    | Patrick Gardner     | 13           |
| CK-12205    | Chloris Kastensmidt | 13           |
| JE-15745    | Joel Eaton          | 13           |

---




---

# Product Analysis 

This section explores product performance through metrics such as sales quantity, profit contribution, and profitability across different categories.

---

## 1. Top Selling Products by Order Quantity

```sql
SELECT 
    t2.`product name` AS `Product Name`,
    t2.`Category` AS `Product Category`,
    SUM(quantity) AS `Quantity`
FROM `superstore` AS t1
LEFT JOIN `product lookup` AS t2 ON t1.`product id` = t2.`product id`
GROUP BY `Product Name`, `Product Category`
ORDER BY `Quantity` DESC
LIMIT 10;
```

**Insight**: Identifies the most frequently ordered products across all categories.

---

## 2. Top Selling Products by Profit

```sql
SELECT 
    t2.`product name` AS `Product Name`,
    t2.`category` AS `Product Category`,
    ROUND(SUM(`Profit`), 2) AS `Total Profit`
FROM `superstore` AS t1
LEFT JOIN `product lookup` AS t2 ON t1.`product id` = t2.`product id`
WHERE t1.`profit or loss` = "Profit"
GROUP BY `Product Name`, `Product Category`
ORDER BY `Total Profit` DESC
LIMIT 10;
```

**Insight**: Highlights products that generate the highest profits.

---

## 3. Total Quantity of Products Sold by Category

```sql
SELECT 
    `pl`.`category` AS `Category`,
    SUM(`quantity`) AS `Total Quantity`
FROM `superstore` `ss`
JOIN `product lookup` `pl` ON `ss`.`product id` = `pl`.`product id`
GROUP BY `Category`
ORDER BY `Total Quantity` DESC;
```

**Insight**: Shows product category contributions by volume of items sold.

---

## 4. Top Selling Products by Quantity in Each Category

```sql
WITH cte AS (
    SELECT 
        `t2`.`category` AS `Category`, 
        `t2`.`product name` AS `Product Name`, 
        `t1`.`quantity` AS `Quantity`, 
        ROW_NUMBER() OVER (PARTITION BY `Category` ORDER BY `Quantity` DESC) AS `Rank`
    FROM `superstore` AS `t1` 
    LEFT JOIN `product lookup` AS `t2` ON `t1`.`product id` = `t2`.`product id`
)
SELECT `Category`, `Product Name`, `Quantity`
FROM cte 
WHERE `Rank` < 4;
```

**Insight**: Retrieves the top 3 best-selling products per category based on quantity.

---

## 5. Product-Level Profit Margin Analysis

```sql
WITH cte AS (
    SELECT 
        `product id` AS `Product Id`, 
        SUM(`profit`) AS `Profit`, 
        SUM(`sales`) AS `Sales`, 
        ROUND((SUM(`profit`)/SUM(`sales`)*100), 2) AS `Profit Margin in %`
    FROM `superstore`
    WHERE `profit or loss` = "Profit"
    GROUP BY `Product Id`
    ORDER BY `Profit Margin in %` DESC
)
SELECT 
    `ct`.`Product Id`, 
    `pl`.`product name` AS `Product Name`, 
    `ct`.`profit margin in %` AS `Profit Margin in %`
FROM `cte` AS `ct`
JOIN `product lookup` AS `pl` ON `ct`.`product id` = `pl`.`product id`
LIMIT 12;
```

**Insight**: Calculates and ranks profit margins at the product level.

---

## 6. Profit/Loss/None Count per Product Category (Pivot View)

```sql
WITH cte AS (
    SELECT 
        DISTINCT `ss`.`order id` AS `Order Id`, 
        `pl`.`category` AS `Product Category`, 
        `ss`.`product id` AS `Product Id`, 
        `ss`.`profit or loss` AS `Profit or Loss or None`
    FROM `superstore` `ss`
    JOIN `product lookup` `pl` ON `ss`.`product id` = `pl`.`product id`
)
SELECT 
    `Product Category`,
    SUM(CASE WHEN `Profit or Loss or None` = "Profit" THEN 1 END) AS `Profit`,
    SUM(CASE WHEN `Profit or Loss or None` = "Loss" THEN 1 END) AS `Loss`,
    SUM(CASE WHEN `Profit or Loss or None` = "None" THEN 1 END) AS `None`
FROM cte
GROUP BY `Product Category`;
```

**Insight**: Breaks down the number of profitable, loss-incurring, and neutral product orders per category.

---




---

# Order Shipment Analysis 

This section examines the shipping duration associated with different shipping modes, providing insights into delivery efficiency and customer experience.

---

## 1. Shipping Days Required by Shipping Mode

```sql
SELECT 
    `ship mode` AS `Ship Mode`, 
    MIN(DATEDIFF(`ship date`, `order date`)) AS `Min Days Required`, 
    MAX(DATEDIFF(`ship date`, `order date`)) AS `Max Days Required`, 
    FLOOR(AVG(DATEDIFF(`ship date`, `order date`))) AS `Avg Shipping Days`
FROM `order lookup`
GROUP BY `Ship Mode`;
```


| Ship Mode      | Min Days Required | Max Days Required | Avg Shipping Days |
| -------------- | ----------------- | ----------------- | ----------------- |
| Standard Class | 3                 | 7                 | 5                 |
| Second Class   | 1                 | 5                 | 3                 |
| First Class    | 1                 | 4                 | 2                 |
| Same Day       | 0                 | 1                 | 0                 |

---

### Insight:

* **Standard Class** takes the longest on average (5 days).
* **Same Day** shipping delivers within 0 to 1 day, as expected.
* **First Class** is the most efficient among premium services.
* Businesses can leverage this insight to optimize shipping options based on customer needs and product types.

